# Sales And Revenue

In [30]:
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score , mean_squared_error,mean_absolute_error

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor

# Creating Baseline Model

In [2]:
df = pd.read_csv("sales_dataset.csv")
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Order ID      1194 non-null   object
 1   Amount        1194 non-null   int64 
 2   Profit        1194 non-null   int64 
 3   Quantity      1194 non-null   int64 
 4   Category      1194 non-null   object
 5   Sub-Category  1194 non-null   object
 6   PaymentMode   1194 non-null   object
 7   Order Date    1194 non-null   object
 8   CustomerName  1194 non-null   object
 9   State         1194 non-null   object
 10  City          1194 non-null   object
 11  Year-Month    1194 non-null   object
dtypes: int64(3), object(9)
memory usage: 112.1+ KB


,Amount,Profit,Quantity
count,1194.000000,1194.000000,1194.000000
mean,5178.089615,1348.992462,10.674204
std,2804.921955,1117.992573,5.777102
min,508.000000,50.000000,1.000000
25%,2799.000000,410.000000,6.000000
50%,5152.000000,1014.000000,11.000000
75%,7626.000000,2035.000000,16.000000
max,9992.000000,4930.000000,20.000000


In [3]:
df = df.drop(columns=["Order ID","Sub-Category","Order Date","CustomerName","Year-Month"])

In [6]:
categorical_cols = df.select_dtypes(include = ["object"]).columns
numerical_cols = df.select_dtypes(include = ["int","float64"]).columns

In [7]:
categorical_cols

Index(['Category', 'PaymentMode', 'State', 'City'], dtype='object')

In [8]:
numerical_cols

Index(['Amount', 'Profit', 'Quantity'], dtype='object')

In [9]:
df.head()

,Amount,Profit,Quantity,Category,PaymentMode,State,City
0,9726,1275,5,Electronics,UPI,Florida,Miami
1,9726,1275,5,Electronics,UPI,Illinois,Chicago
2,9726,1275,5,Electronics,UPI,New York,Buffalo
3,4975,1330,14,Electronics,UPI,Florida,Miami
4,4975,1330,14,Electronics,UPI,Illinois,Chicago


In [10]:
# Encoding
from sklearn.preprocessing import OneHotEncoder

cols = ['Category', 'PaymentMode', 'State', 'City']

ohe = OneHotEncoder(drop = "first",sparse_output = False, handle_unknown = "ignore")
encoded = ohe.fit_transform(df[cols])
encoded_df = pd.DataFrame(encoded,columns = ohe.get_feature_names_out(cols),index = df.index)

#Append original dataframe
df = pd.concat([df.drop(columns = cols),encoded_df],axis = 1)

In [11]:
# Split features and target
X = df.drop("Profit",axis = 1)
y = df["Profit"]

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size = 0.2, random_state = 42
)

In [12]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
X_train_scaled

array([[-1.25852384, -0.28738984, -0.70488666, ..., -0.27688068,
        -0.25194124, -0.21713849],
       [-1.20447112, -1.15537621, -0.70488666, ..., -0.27688068,
        -0.25194124, -0.21713849],
       [ 1.58031065, -1.50257076, -0.70488666, ..., -0.27688068,
        -0.25194124, -0.21713849],
       ...,
       [-0.28593053, -0.98177894,  1.41866779, ..., -0.27688068,
        -0.25194124, -0.21713849],
       [ 1.01311273,  0.40699927, -0.70488666, ..., -0.27688068,
        -0.25194124, -0.21713849],
       [-0.5981561 , -1.32897349, -0.70488666, ..., -0.27688068,
        -0.25194124, -0.21713849]])

In [14]:
X_test_scaled

array([[-0.7905411 , -0.80818166,  1.41866779, ..., -0.27688068,
        -0.25194124, -0.21713849],
       [-1.41712589,  1.10138837,  1.41866779, ..., -0.27688068,
        -0.25194124, -0.21713849],
       [ 0.7126223 , -0.80818166, -0.70488666, ..., -0.27688068,
        -0.25194124, -0.21713849],
       ...,
       [ 1.45300228,  0.92779109, -0.70488666, ..., -0.27688068,
        -0.25194124, -0.21713849],
       [ 0.07821409,  0.92779109, -0.70488666, ..., -0.27688068,
        -0.25194124, -0.21713849],
       [ 0.59811588,  0.92779109, -0.70488666, ..., -0.27688068,
        -0.25194124, -0.21713849]])

In [40]:
# Liner Regression
lin_reg = LinearRegression()

lin_reg.fit(X_train_scaled,y_train)
y_pred_test = lin_reg.predict(X_test_scaled)

# Sales & profit data always have outliers So MAE is safer.
# If your target is money → MAE is the most important metric

print("Linear Regression MAE: ",mean_absolute_error(y_test,y_pred_test))
print("Linear Regression r^2: ",r2_score(y_test,y_pred_test)* 100,"%")
print("Linear Regression RMSE: ",np.sqrt(mean_squared_error(y_test,y_pred_test)))

Linear Regression MAE:  662.0081354252015
Linear Regression r^2:  41.81506130519396 %
Linear Regression RMSE:  858.6166969613125


# Model - 2 KNN

In [39]:
# KNN = KNeighborsRegressor

knn_reg = KNeighborsRegressor(
    n_neighbors = 5,
    metric = "euclidean",
    p = 5
)

knn_reg.fit(X_train_scaled,y_train)
y_pred_test = knn_reg.predict(X_test_scaled)

print("KNN Regressoion MAE:", mean_absolute_error(y_test,y_pred_test))
print("KNN Regressoion: r^2", r2_score(y_test,y_pred_test)* 100,"%")
print("KNN Regressoion RMSE:", np.sqrt(mean_squared_error(y_test,y_pred_test)))

KNN Regressoion MAE: 893.6828451882844
KNN Regressoion: r^2 -6.043245628675686 %
KNN Regressoion RMSE: 1159.139071298556


# Random Forest (Ensemble Learning)

In [38]:
rf = RandomForestRegressor(
    n_estimators = 200,
    max_depth = None,
    random_state = 42
)

rf.fit(X_train,y_train)
y_pred_rf = rf.predict(X_test)

print("RandomForest Regressor MAE: ",mean_absolute_error(y_test,y_pred_rf))
print("RandomForest Regressor r^2: ",r2_score(y_test,y_pred_rf) * 100,"%")
print("RandomForest Regressor RMSE: ",np.sqrt(mean_squared_error(y_test,y_pred_rf)))

RandomForest Regressor MAE:  558.1159414225941
RandomForest Regressor r^2:  51.32445556564067 %
RandomForest Regressor RMSE:  785.3250452860627


# Gradient Boosting (Ensemble Learning)

In [41]:
gb = GradientBoostingRegressor(
    n_estimators = 150,
    learning_rate=0.1,
    max_depth = 3,
    random_state = 42
)

gb.fit(X_train,y_train)
y_pred_gb = gb.predict(X_test)

print("GradientBoosting Regressor MAE: ",mean_absolute_error(y_test,y_pred_gb))
print("GradientBoosting Regressor r^2: ",r2_score(y_test,y_pred_gb)* 100,"%")
print("GradientBoosting Regressor RMSE: ",np.sqrt(mean_squared_error(y_test,y_pred_gb)))

GradientBoosting Regressor MAE:  598.1211070123037
GradientBoosting Regressor r^2:  47.389956992945784 %
GradientBoosting Regressor RMSE:  816.4476955082688


# Voting Classifier (Ensemble Lerning)

In [42]:
Voting_rg = VotingRegressor(
    estimators = [
        ("lr", LinearRegression(n_jobs = 5, fit_intercept=True)),
        ("knn", KNeighborsRegressor(n_neighbors=5)),
        ("rf", RandomForestRegressor(n_estimators = 200,random_state = 42))
    ],
    weights=None
)

Voting_rg.fit(X_train_scaled,y_train)
y_pred_vote = Voting_rg.predict(X_test_scaled)

print("Voting Regressor MAE: ",mean_absolute_error(y_test,y_pred_vote))
print("Voting Regressor r^2: ",r2_score(y_test,y_pred_vote)* 100,"%")
print("Voting Regressor RMSE: ",np.sqrt(mean_squared_error(y_test,y_pred_vote)))

Voting Regressor MAE:  661.9550629378443
Voting Regressor r^2:  41.252241705354606 %
Voting Regressor RMSE:  862.7593783071162


# Result

| Model               | Mean Absolute Error(MAE)  |
|---------------------|:-------------------------:|
| Linear Regression   |         662.00            |
| KNN Regressor       |         893.68            |
| Random Forest       |         558.11            | 
| Gredient Boosting   |         598.12            |
| Voting Regressor    |         661.95            |

# Best Regressor that we should use for Sales And Revenue (Mean Absolute Error(MAE)) - Random Forest with r2_score of 51.32%